# Geração Consistente de Resultados — TCC2 Ciclones

Reproduz **todos** os resultados do TCC a partir dos dados brutos TCIR e dos modelos `.keras`.

## Saídas geradas
| Arquivo | Usado em |
|---|---|
| `plots/{modelo}_scatter.png` | Tabela 3 (MSE / RMSE / MAE com TTA) |
| `errors/{modelo}/errors.csv` | Wilcoxon e Shapiro-Wilk (**com TTA**) |
| `scatter_metrics.json` | Tabela 3 (valores numéricos) |
| `wilcoxon_summary.json` | Tabela 4 |
| `shapiro_wilk_summary.json` | Justificativa do Wilcoxon |
| `shap/shap_global_mean.png` | Figura de importância global SHAP |
| `shap/shap_samples_grid.png` | Grade de amostras por intensidade |
| `shap/shap_error_analysis.png` | Análise de erro × magnitude SHAP |
| `shap/shap_results.h5` | Valores SHAP brutos (opcional) |

## Consistência entre Tabelas 3 e 4
Tanto o scatter plot (Tabela 3) quanto o `errors.csv` (Tabelas 4 e 5) usam **TTA com 10 rotações**.
Assim os erros reportados são os mesmos em todas as tabelas.

## Alinhamento de amostras
| Modelo | Pré-proc | REMOVE_FROM_TC |
|---|---|---|
| original | nenhum | 2 |
| tcc_I | Diff1 | 1 |
| tcc_II / resnet / mobilenet_v2 | Diff2 | 0 |

> **Antes de rodar:** ajuste `TCIR_DIR`, `MODELS_DIR` e `SHAP_MODEL` na célula de Configuração.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'tables', 'dask', 'shap'], check=False)

import os, json, warnings, gc
import numpy as np
import pandas as pd
import h5py
import dask.array as da
import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import wilcoxon, shapiro, spearmanr
from pathlib import Path

warnings.filterwarnings('ignore')
keras = tf.keras
print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# =============================================================================
# CONFIGURAÇÃO — ajuste os caminhos para seus datasets do Kaggle
# =============================================================================

TCIR_DIR   = '/kaggle/input/tcir-ciclone-dataset'
MODELS_DIR = '/kaggle/input/cyclone-trained-models'
OUTPUT_DIR = '/kaggle/working'

CHANNELS    = [0, 3]
IMG_W       = 64
N_ROTATIONS = 10
BATCH_SIZE  = 64

SHAP_MODEL = 'tcc_II'
SHAP_N     = 500
SHAP_BG    = 100
SHAP_BATCH = 8
SHAP_SEED  = 42

def _rot_w(w):
    rw = int(np.ceil(np.sqrt(w**2 * 2)))
    return rw if rw % 2 == 0 else rw + 1

ROTATION_WIDTH = _rot_w(IMG_W)  # 92

# img_width: tamanho de entrada na inferência.
# CNN-TC (original, tcc_I, tcc_II) → 64x64
# ResNet50 e MobileNetV2 → 224x224 (treinados com resize para 224)
MODEL_CONFIGS = {
    'original': {
        'model_path':     f'{MODELS_DIR}/result/original/model.keras',
        'remove_from_tc': 2, 'diff_order': 0, 'img_width': 64,
        'label': 'Original (CNN-TC)',
    },
    'tcc_I': {
        'model_path':     f'{MODELS_DIR}/result/tcc_I/model.keras',
        'remove_from_tc': 1, 'diff_order': 1, 'img_width': 64,
        'label': 'Diff1 (tcc_I)',
    },
    'tcc_II': {
        'model_path':     f'{MODELS_DIR}/result/tcc_II/2/model.keras',
        'remove_from_tc': 0, 'diff_order': 2, 'img_width': 64,
        'label': 'Diff2 (tcc_II)',
    },
    'resnet': {
        'model_path':     f'{MODELS_DIR}/result/resnet/tcc_ii_preprocess/model.keras',
        'remove_from_tc': 0, 'diff_order': 2, 'img_width': 224,
        'label': 'ResNet50',
    },
    'mobilenet_v2': {
        'model_path':     f'{MODELS_DIR}/result/mobilenetv2/model.keras',
        'remove_from_tc': 0, 'diff_order': 2, 'img_width': 224,
        'label': 'MobileNetV2',
    },
}

print(f'ROTATION_WIDTH = {ROTATION_WIDTH}')
print(f'SHAP_MODEL     = {SHAP_MODEL}  |  SHAP_N = {SHAP_N}')
print('Status dos modelos:')
for name, cfg in MODEL_CONFIGS.items():
    status = '✓' if os.path.exists(cfg['model_path']) else '⚠️  NÃO ENCONTRADO'
    print(f'  {name:<18} {status}  img_width={cfg["img_width"]}')

In [ ]:
# =============================================================================
# FUNÇÕES DE PRÉ-PROCESSAMENTO (lógica inline de data.py)
# =============================================================================

def load_raw_data(tcir_dir):
    files = [
        f'{tcir_dir}/TCIR-ATLN_EPAC_WPAC.h5',
        f'{tcir_dir}/TCIR-ALL_2017.h5',
        f'{tcir_dir}/TCIR-CPAC_IO_SH.h5',
    ]
    handles = [h5py.File(f, mode='r') for f in files]
    arrays  = [da.from_array(h['matrix']) for h in handles]
    infos   = [pd.read_hdf(f, key='info', mode='r') for f in files]
    return da.concatenate(arrays, axis=0), pd.concat(infos).reset_index(drop=True), handles

def filter_years(data, info, y_start, y_end):
    years = np.array([datetime.datetime.strptime(t, '%Y%m%d%H').year for t in info['time']])
    idx   = np.where((years >= y_start) & (years <= y_end))[0]
    return data[idx], info.iloc[idx].reset_index(drop=True)

def cut_images(images, width):
    s = images.shape[1] // 2 - width // 2
    e = images.shape[1] // 2 + width // 2
    return images[:, s:e, s:e, :]

def clear_images(images):
    images = da.nan_to_num(images)
    images[images > 1000] = 0
    return images

def compute_norm_stats(trainds_lazy, channels):
    means = [float(da.mean(trainds_lazy[:, :, :, c]).compute()) for c in channels]
    stds  = [float(da.std(trainds_lazy[:, :, :, c]).compute())  for c in channels]
    return means, stds

def normalize(images, means, stds):
    for ci in range(len(means)):
        images[:, :, :, ci] = (images[:, :, :, ci] - means[ci]) / stds[ci]
    return images

def build_diff1_test(images, info):
    """Diff1: |I_n - I_{n-1}| no canal 0. Remove 1ª imagem por ciclone."""
    imgs_out, info_out = [], []
    for _, grp in info.groupby('ID'):
        grp = grp.sort_values('time')
        idx = grp.index.to_numpy()
        if len(idx) < 2:
            continue
        imgs = images[idx]
        diff = np.abs(imgs[1:, :, :, :1] - imgs[:-1, :, :, :1])
        imgs_out.append(np.concatenate([imgs[1:], diff], axis=-1))
        info_out.append(grp.iloc[1:])
    return np.concatenate(imgs_out, axis=0), pd.concat(info_out).reset_index(drop=True)

def build_diff2_test(images, info):
    """Diff2: |I_n - 2*I_{n-1} + I_{n-2}| no canal 0. Remove 2 primeiras por ciclone."""
    imgs_out, info_out = [], []
    for _, grp in info.groupby('ID'):
        grp = grp.sort_values('time')
        idx = grp.index.to_numpy()
        if len(idx) < 3:
            continue
        imgs = images[idx]
        diff = np.abs(imgs[2:, :, :, :1] - 2.0*imgs[1:-1, :, :, :1] + imgs[:-2, :, :, :1])
        imgs_out.append(np.concatenate([imgs[2:], diff], axis=-1))
        info_out.append(grp.iloc[2:])
    return np.concatenate(imgs_out, axis=0), pd.concat(info_out).reset_index(drop=True)

print('Funções de pré-processamento definidas.')

In [ ]:
# =============================================================================
# CARGA E NORMALIZAÇÃO DOS DADOS
# =============================================================================

print('Carregando dados TCIR...')
data, info, _handles = load_raw_data(TCIR_DIR)
print(f'Total de amostras: {len(info)}')

print('Calculando estatísticas de normalização (dados de treino 2003-2014)...')
train_lazy, _ = filter_years(data, info, 2003, 2014)
train_lazy     = cut_images(clear_images(train_lazy), ROTATION_WIDTH)
means, stds    = compute_norm_stats(train_lazy, CHANNELS)
print(f'Canal {CHANNELS[0]}: µ={means[0]:.6f}  σ={stds[0]:.6f}')
print(f'Canal {CHANNELS[1]}: µ={means[1]:.6f}  σ={stds[1]:.6f}')

print('\nCarregando e normalizando dados de teste (2017)...')
test_lazy, testinfo = filter_years(data, info, 2017, 2017)
test_lazy = cut_images(clear_images(test_lazy), ROTATION_WIDTH)
testds    = normalize(test_lazy.compute()[:, :, :, CHANNELS].astype(np.float32), means, stds)

print(f'Shape: {testds.shape}  |  Ciclones: {testinfo["ID"].nunique()}')

In [ ]:
# =============================================================================
# CONSTRUÇÃO DOS DATASETS DE TESTE POR TIPO DE MODELO
# =============================================================================

test_data = {}
test_data['original'] = (testds.copy(), testinfo.copy())

imgs_d1, info_d1 = build_diff1_test(testds, testinfo)
test_data['tcc_I'] = (imgs_d1, info_d1)

imgs_d2, info_d2 = build_diff2_test(testds, testinfo)
for k in ['tcc_II', 'resnet', 'mobilenet_v2']:
    test_data[k] = (imgs_d2, info_d2.copy())

n_total    = testds.shape[0]
n_cyclones = testinfo['ID'].nunique()
aligned_n  = n_total - 2 * n_cyclones

print(f'Total={n_total}  Ciclones={n_cyclones}  Alinhado esperado={aligned_n}')
print('Verificação:')
for name, cfg in MODEL_CONFIGS.items():
    imgs, inf = test_data[name]
    r     = cfg['remove_from_tc']
    after = imgs.shape[0] - r * inf['ID'].nunique()
    ok    = '✓' if after == aligned_n else '⚠️'
    print(f'  {name:<18} {ok}  shape={imgs.shape}  após remove={after}  img_width={cfg["img_width"]}')

In [ ]:
# =============================================================================
# FUNÇÕES DE INFERÊNCIA E VISUALIZAÇÃO
# =============================================================================

def _rotate_batch(batch_np, angle_deg, out_size):
    """Rotaciona batch inteiro via GPU e faz crop/pad para out_size×out_size."""
    B, H, W = batch_np.shape[0], batch_np.shape[1], batch_np.shape[2]
    ar = float(angle_deg * np.pi / 180.0)
    cx, cy = W / 2.0, H / 2.0
    ca, sa = np.cos(ar), np.sin(ar)
    t = np.array([ca, -sa, (1-ca)*cx+sa*cy, sa, ca, (1-ca)*cy-sa*cx, 0., 0.], np.float32)
    rotated = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.cast(batch_np, tf.float32),
        transforms=tf.constant(np.tile(t[None, :], (B, 1))),
        output_shape=[H, W],
        interpolation='BILINEAR', fill_mode='REFLECT', fill_value=0.0,
    )
    if out_size != H or out_size != W:
        rotated = tf.image.resize_with_crop_or_pad(rotated, out_size, out_size)
    return rotated.numpy()


def _batch_predict(model, images_np, batch_size):
    preds = []
    for s in range(0, len(images_np), batch_size):
        p = model.predict(images_np[s:s+batch_size], verbose=0)
        preds.extend((p[:, 0] if p.ndim > 1 else p).tolist())
    return np.array(preds, dtype=np.float32)


def predict_tta(model, images, img_width, n_rotations=N_ROTATIONS, batch_size=BATCH_SIZE):
    """TTA (Rotation Blending): média de N rotações uniformes em [0, 360)."""
    angles   = np.linspace(0, 360, n_rotations, endpoint=False)
    N        = len(images)
    pred_sum = np.zeros(N, dtype=np.float32)
    for ri, angle in enumerate(angles):
        rot_preds = []
        for s in range(0, N, batch_size):
            rotated = _rotate_batch(images[s:s+batch_size].astype(np.float32), angle, img_width)
            p = model.predict(rotated, verbose=0)
            rot_preds.extend((p[:, 0] if p.ndim > 1 else p).tolist())
        pred_sum += np.array(rot_preds, dtype=np.float32)
        print(f'      Rotação {ri+1}/{n_rotations} ({angle:.0f}°)')
    return pred_sum / n_rotations


def apply_remove_from_tc(info_df, vmax_arr, preds_arr, remove_from_tc):
    """Remove N primeiras imagens de cada ciclone para alinhar todos os modelos."""
    df = pd.DataFrame({
        'ID_Ciclone': info_df['ID'].values,
        'Data_Hora':  info_df['time'].values,
        'Vmax': vmax_arr, 'Pred': preds_arr,
    }).sort_values(['ID_Ciclone', 'Data_Hora'])
    if remove_from_tc > 0:
        df = (
            df.groupby('ID_Ciclone', group_keys=False)
              .apply(lambda x: x.iloc[remove_from_tc:])
              .reset_index(drop=True)
        )
    else:
        df = df.reset_index(drop=True)
    df['Erro_Bruto']    = df['Vmax'] - df['Pred']
    df['Erro_Absoluto'] = df['Erro_Bruto'].abs()
    return df


def plot_scatter(df, label, save_path):
    """Scatter plot predições TTA vs valores reais com MSE, RMSE e MAE."""
    real, preds = df['Vmax'].values, df['Pred'].values
    mse  = mean_squared_error(real, preds)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(real, preds)

    fig, ax = plt.subplots(figsize=(8, 8), dpi=100)
    ax.scatter(real, preds, alpha=0.5, s=25, color='steelblue', label='Predições')
    lo = min(real.min(), preds.min()) - 5
    hi = max(real.max(), preds.max()) + 5
    ax.plot([lo, hi], [lo, hi], 'r--', lw=2, label='y = x')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
    ax.set_xlabel('Velocidade real (nós)', fontsize=12)
    ax.set_ylabel('Velocidade predita (nós)', fontsize=12)
    ax.set_title(
        f'{label}\nMSE: {mse:.2f}  |  RMSE: {rmse:.2f} nós  |  MAE: {mae:.2f} nós  |  N={len(real)}',
        fontsize=13, fontweight='bold'
    )
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'      → {save_path}  MSE={mse:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')
    return float(mse), float(rmse), float(mae), int(len(real))


print('Funções de inferência definidas.')

In [ ]:
# =============================================================================
# LOOP PRINCIPAL: por modelo → TTA → errors.csv + scatter plot
#
# Uma única passagem TTA é usada para ambas as saídas:
#   • errors.csv  → Tabelas 4 e 5 (Wilcoxon e Shapiro-Wilk)
#   • scatter plot → Tabela 3 (MSE / RMSE / MAE)
# Isso garante que os erros reportados sejam os mesmos em todas as tabelas.
# =============================================================================

os.makedirs(f'{OUTPUT_DIR}/plots', exist_ok=True)
all_metrics = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f'\n{"="*65}')
    print(f'  {model_name}  —  {cfg["label"]}')
    print(f'{"="*65}')

    if not os.path.exists(cfg['model_path']):
        print(f'  ⚠️  model.keras não encontrado — pulando.')
        continue

    model = keras.models.load_model(cfg['model_path'], compile=False)
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])

    imgs, info_df = test_data[model_name]
    vmax   = info_df['Vmax'].values.astype(np.float32)
    remove = cfg['remove_from_tc']
    iw     = cfg['img_width']

    # ── TTA (10 rotações) → predições usadas em TUDO ──
    print(f'  TTA ({N_ROTATIONS} rotações, img_width={iw})...')
    preds_tta = predict_tta(model, imgs, iw)
    df_tta    = apply_remove_from_tc(info_df, vmax, preds_tta, remove)

    # ── errors.csv (Tabelas 4 e 5) ──
    err_dir = f'{OUTPUT_DIR}/errors/{model_name}'
    os.makedirs(err_dir, exist_ok=True)
    df_tta[['ID_Ciclone', 'Data_Hora', 'Erro_Bruto', 'Erro_Absoluto']].to_csv(
        f'{err_dir}/errors.csv', index=False
    )
    print(f'    errors.csv → {len(df_tta)} amostras')

    # ── scatter plot (Tabela 3) ──
    mse, rmse, mae, n = plot_scatter(
        df_tta, cfg['label'], f'{OUTPUT_DIR}/plots/{model_name}_scatter.png'
    )
    all_metrics[model_name] = {'mse': mse, 'rmse': rmse, 'mae': mae, 'n': n}

    del model; keras.backend.clear_session(); gc.collect()

print(f'\n{"="*65}')
print(f'  {"Modelo":<18} {"MSE":>8} {"RMSE":>8} {"MAE":>8} {"N":>6}')
print('  ' + '-'*50)
for name, m in all_metrics.items():
    print(f'  {name:<18} {m["mse"]:>8.4f} {m["rmse"]:>8.4f} {m["mae"]:>8.4f} {m["n"]:>6}')
print(f'{"="*65}')

with open(f'{OUTPUT_DIR}/scatter_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

In [ ]:
# =============================================================================
# MAPAS DE EXPLICABILIDADE — SHAP (GradientExplainer)
# =============================================================================

shap_cfg = MODEL_CONFIGS[SHAP_MODEL]
shap_out = f'{OUTPUT_DIR}/shap'
os.makedirs(shap_out, exist_ok=True)

if not os.path.exists(shap_cfg['model_path']):
    print(f'⚠️  Modelo SHAP não encontrado — célula ignorada.')
else:
    print(f'Carregando modelo {SHAP_MODEL} para SHAP...')
    shap_model = keras.models.load_model(shap_cfg['model_path'], compile=False)
    input_size = shap_model.input_shape[1]
    print(f'  Input shape: {shap_model.input_shape}')

    shap_imgs_all, shap_info_all = test_data[SHAP_MODEL]
    if shap_imgs_all.shape[1] != input_size:
        shap_imgs_all = tf.image.resize_with_crop_or_pad(
            tf.cast(shap_imgs_all, tf.float32), input_size, input_size
        ).numpy()

    rng  = np.random.default_rng(SHAP_SEED)
    N    = len(shap_imgs_all)
    idx  = np.sort(rng.choice(N, min(SHAP_N, N), replace=False))
    shap_imgs = shap_imgs_all[idx].astype(np.float32)
    shap_info = shap_info_all.iloc[idx].reset_index(drop=True)
    print(f'  Amostras: {len(shap_imgs)}  shape={shap_imgs.shape}')

    bg_data = shap_imgs[rng.choice(len(shap_imgs), min(SHAP_BG, len(shap_imgs)), replace=False)]
    print(f'  Background: {bg_data.shape}')

    print('\nInicializando GradientExplainer...')
    explainer = shap.GradientExplainer(shap_model, bg_data)

    print(f'Calculando SHAP values ({len(shap_imgs)} amostras, batch={SHAP_BATCH})...')
    shap_maps_list, preds_list = [], []
    for start in range(0, len(shap_imgs), SHAP_BATCH):
        batch  = shap_imgs[start:start+SHAP_BATCH]
        sv_raw = explainer.shap_values(batch)
        sv     = np.array(sv_raw[0] if isinstance(sv_raw, list) else sv_raw, dtype=np.float32)
        if sv.ndim == 5 and sv.shape[-1] == 1:
            sv = sv[..., 0]
        p = shap_model.predict(batch, verbose=0)
        preds_list.append((p[:, 0] if p.ndim > 1 else p).astype(np.float32))
        shap_maps_list.append(sv)
        done = min(start + SHAP_BATCH, len(shap_imgs))
        print(f'  [{done:4d}/{len(shap_imgs)}]  {100*done/len(shap_imgs):.1f}%', flush=True)

    shap_maps  = np.concatenate(shap_maps_list, axis=0)
    preds_shap = np.concatenate(preds_list,     axis=0)
    print(f'\nSHAP maps: {shap_maps.shape}')

    # ── Salvar HDF5 ──
    out_h5 = f'{shap_out}/shap_results.h5'
    with h5py.File(out_h5, 'w') as f:
        f.create_dataset('images',      data=shap_imgs,  compression='gzip')
        f.create_dataset('shap_values', data=shap_maps,  compression='gzip')
        f.create_dataset('predictions', data=preds_shap, compression='gzip')
        f.attrs['model'] = SHAP_MODEL; f.attrs['n'] = len(shap_imgs)
    shap_info.to_hdf(out_h5, key='info', mode='a', format='table')
    print(f'Salvo: {out_h5}')

    def _norm01(a):
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn + 1e-9)

    vmax_vals = shap_info['Vmax'].values

    # ── Plot 1: importância média global ──
    mean_abs = np.mean(np.abs(shap_maps), axis=0)
    C = mean_abs.shape[-1]
    fig, axes = plt.subplots(1, C + 1, figsize=(5*(C+1), 4.5))
    for ax, ttl, m in zip(axes,
                          [f'Canal {c}' for c in range(C)] + ['Soma canais'],
                          [mean_abs[..., c] for c in range(C)] + [mean_abs.sum(-1)]):
        im = ax.imshow(m, cmap='hot', interpolation='bilinear')
        ax.set_title(ttl, fontsize=10); ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'SHAP — Importância Média |SHAP|  ({SHAP_MODEL}, N={len(shap_imgs)})', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{shap_out}/shap_global_mean.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Salvo: {shap_out}/shap_global_mean.png')

    # ── Plot 2: grade de amostras por intensidade ──
    p25, p50, p75 = np.percentile(vmax_vals, [25, 50, 75])
    sel = [int(np.argmin(np.abs(vmax_vals - p))) for p in [p25, p50, p75]]
    fig, axes = plt.subplots(3, 3, figsize=(12, 10.5))
    for col, ttl in enumerate(['Imagem (Ch 0)', 'SHAP Overlay (signed)', '|SHAP| soma canais']):
        axes[0, col].set_title(ttl, fontsize=10, pad=8)
    for row, (i, rlabel) in enumerate(zip(sel,
            [f'Baixa (~{p25:.0f} kt)', f'Média (~{p50:.0f} kt)', f'Alta (~{p75:.0f} kt)'])):
        img0 = _norm01(shap_imgs[i][..., 0])
        sv_s = shap_maps[i].sum(-1)
        vclip = float(np.percentile(np.abs(sv_s), 99)) or 1e-6
        info_str = (f'{rlabel}\nPred {preds_shap[i]:.1f} kt | '
                    f'Real {vmax_vals[i]:.1f} kt | Err {abs(preds_shap[i]-vmax_vals[i]):.1f}')
        ax0, ax1, ax2 = axes[row]
        for ax in (ax0, ax1, ax2): ax.axis('off')
        ax0.imshow(img0, cmap='gray')
        ax0.set_ylabel(info_str, fontsize=7, rotation=0, ha='right', va='center')
        ax1.imshow(img0, cmap='gray', alpha=0.55)
        im1 = ax1.imshow(sv_s, cmap='seismic', vmin=-vclip, vmax=vclip, alpha=0.6)
        plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
        im2 = ax2.imshow(np.abs(sv_s), cmap='hot', vmin=0, vmax=vclip)
        plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    fig.suptitle(f'SHAP — Amostras por Intensidade  ({SHAP_MODEL})', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{shap_out}/shap_samples_grid.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Salvo: {shap_out}/shap_samples_grid.png')

    # ── Plot 3: análise de erro × magnitude SHAP ──
    errors_shap = np.abs(preds_shap - vmax_vals)
    shap_mag    = np.mean(np.abs(shap_maps), axis=(1, 2, 3))
    H = shap_maps.shape[1]
    s, e = H // 4, 3 * H // 4
    shap_center = np.mean(np.abs(shap_maps[:, s:e, s:e, :]), axis=(1, 2, 3))
    shap_border = shap_mag - shap_center
    corr, pval  = spearmanr(shap_mag, errors_shap)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sc = axes[0].scatter(shap_mag, errors_shap, c=vmax_vals, cmap='plasma', alpha=0.4, s=10)
    axes[0].set_xlabel('|SHAP| médio'); axes[0].set_ylabel('|Erro| (kt)')
    axes[0].set_title(f'Magnitude SHAP × Erro\nSpearman r={corr:.3f}  p={pval:.2e}')
    plt.colorbar(sc, ax=axes[0]).set_label('Vmax real (kt)')
    bins = np.histogram_bin_edges(errors_shap, bins=8)
    bc   = 0.5*(bins[:-1]+bins[1:]); w = (bins[1]-bins[0])*0.38
    dig  = np.digitize(errors_shap, bins)
    cm_  = [shap_center[dig==k+1].mean() if (dig==k+1).any() else 0 for k in range(len(bc))]
    bm_  = [shap_border[dig==k+1].mean() if (dig==k+1).any() else 0 for k in range(len(bc))]
    axes[1].bar(bc-w/2, cm_, width=w, color='#4e9af1', label='Centro', alpha=0.85)
    axes[1].bar(bc+w/2, bm_, width=w, color='#f1724e', label='Borda',  alpha=0.85)
    axes[1].set_xlabel('|Erro| (kt)'); axes[1].set_ylabel('SHAP médio')
    axes[1].set_title('Concentração SHAP: Centro vs. Borda'); axes[1].legend()
    mae_s = np.mean(errors_shap); rmse_s = np.sqrt(np.mean(errors_shap**2))
    fig.suptitle(f'SHAP — Análise de Erro  (MAE={mae_s:.2f} kt  RMSE={rmse_s:.2f} kt)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{shap_out}/shap_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Salvo: {shap_out}/shap_error_analysis.png')

    del shap_model; keras.backend.clear_session(); gc.collect()
    print('\n✓ SHAP concluído.')

In [ ]:
# =============================================================================
# TESTE DE WILCOXON SIGNED-RANK
# =============================================================================

errors_dir  = Path(f'{OUTPUT_DIR}/errors')
df_original = pd.read_csv(errors_dir / 'original' / 'errors.csv')
print(f'Original: {len(df_original)} amostras')

comparisons = {
    'tcc_I': 'Diff1 (tcc_I)', 'tcc_II': 'Diff2 (tcc_II)',
    'resnet': 'ResNet50', 'mobilenet_v2': 'MobileNetV2',
}
w_results = []
print('\n' + '='*70)
print('  WILCOXON — Original vs Todos')
print('='*70)

for key, label in comparisons.items():
    path = errors_dir / key / 'errors.csv'
    if not path.exists():
        print(f'  ⚠️  {key}/errors.csv ausente — pulando.')
        continue
    df_other = pd.read_csv(path)
    merged   = df_original.merge(df_other, on=['ID_Ciclone', 'Data_Hora'],
                                  suffixes=('_orig', '_other'))
    if len(merged) < len(df_original):
        print(f'  ⚠️  Alinhamento parcial: {len(merged)}')
    eo = merged['Erro_Absoluto_orig'].values
    em = merged['Erro_Absoluto_other'].values
    v  = ~(np.isnan(eo) | np.isnan(em))
    eo, em = eo[v], em[v]
    stat, p = wilcoxon(eo, em, alternative='two-sided')
    pct     = round(100 * (em < eo).sum() / len(eo), 1)
    r = {
        'model': label, 'n': int(len(eo)),
        'mean_original': float(np.mean(eo)), 'mean_other': float(np.mean(em)),
        'median_original': float(np.median(eo)), 'median_other': float(np.median(em)),
        'pct_better': pct, 'statistic': float(stat),
        'p_value': float(p), 'significant': bool(p < 0.05),
    }
    w_results.append(r)
    print(f'\n  Original vs {label}')
    print(f'    N={r["n"]}  µ_orig={r["mean_original"]:.4f}  µ_mod={r["mean_other"]:.4f}')
    print(f'    % melhor={pct}%  W={stat:.2f}  p={p:.6f}  Sig: {"✓ SIM" if p<0.05 else "✗ NÃO"}')

with open(f'{OUTPUT_DIR}/wilcoxon_summary.json', 'w') as f:
    json.dump(w_results, f, indent=2)

print('\n' + '='*70)
print(f'{"Modelo":<22} {"µ Orig":>8} {"µ Mod":>8} {"% Melhor":>10} {"p-value":>12} {"Sig?":>5}')
print('-'*70)
for r in w_results:
    print(f'{r["model"]:<22} {r["mean_original"]:>8.4f} {r["mean_other"]:>8.4f} '
          f'{r["pct_better"]:>9.1f}% {r["p_value"]:>12.6f} {"Sim" if r["significant"] else "Não":>5}')
print('='*70)
print('\n✓ wilcoxon_summary.json salvo.')

In [ ]:
# =============================================================================
# TESTE DE SHAPIRO-WILK
# =============================================================================

print('='*70)
print('  SHAPIRO-WILK')
print('='*70)
print(f'{"Modelo":<18} {"Métrica":<22} {"N":>6} {"W":>8} {"p-value":>12} {"Normal?":>8}')
print('-'*70)

sw_results = []
for model_name in ['original'] + list(comparisons.keys()):
    path = errors_dir / model_name / 'errors.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path)
    for col in ['Erro_Bruto', 'Erro_Absoluto']:
        vals = df[col].dropna().values
        W, p = shapiro(vals)
        is_n = bool(p > 0.05)
        sw_results.append({'model': model_name, 'metric': col, 'n': int(len(vals)),
                           'W': float(W), 'p_value': float(p), 'is_normal': is_n})
        p_str = f'{p:.3e}' if p < 0.001 else f'{p:.4f}'
        print(f'{model_name:<18} {col:<22} {len(vals):>6} {W:>8.4f} {p_str:>12} {"Sim" if is_n else "Não":>8}')

with open(f'{OUTPUT_DIR}/shapiro_wilk_summary.json', 'w') as f:
    json.dump(sw_results, f, indent=2)

print('\nConclusão: p < 0.05 em todos → erros não-normais → justifica Wilcoxon.')
print('✓ shapiro_wilk_summary.json salvo.')

In [ ]:
# =============================================================================
# RESUMO DOS ARQUIVOS GERADOS
# =============================================================================

print('#'*70)
print('  ARQUIVOS GERADOS EM /kaggle/working/')
print('#'*70)
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    print(f'{"  "*level}{os.path.basename(root)}/')
    for fname in sorted(files):
        size = os.path.getsize(os.path.join(root, fname))
        print(f'{"  "*(level+1)}{fname}  ({size/1024:.1f} KB)')
print('#'*70)

## Como configurar os inputs no Kaggle

### Dataset TCIR
Crie um dataset Kaggle com os 3 arquivos HDF5 e atualize `TCIR_DIR`.

### Dataset de modelos
Crie um dataset com a estrutura abaixo e atualize `MODELS_DIR`:
```
result/
  original/model.keras
  tcc_I/model.keras
  tcc_II/2/model.keras
  resnet/tcc_ii_preprocess/model.keras
  mobilenetv2/model.keras
```

### GPU
Habilite **GPU T4 x2** em *Settings → Accelerator*.

### SHAP lento?
Reduza `SHAP_N` para 200 ou menos na célula de configuração.

### Tempo estimado (T4 GPU)
TTA com 10 rotações × 5 modelos × ~4296 amostras ≈ 20–35 min no total.